<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex08.1-stationary-heat/Ex08.1_02_plate_with_hole_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_08.1 · Notebook 02 — The Plate with a Cooling Hole

**Paired with L8.1 · Stationary Heat Transfer**

Now the real geometry. Fixed temperature on the hole, insulated outer edges,
uniform internal generation:

$$T = 0 \text{ on the hole}, \qquad
\frac{\partial T}{\partial n} = 0 \text{ on the outer edges},
\qquad \frac{Q}{k} = \text{const}$$

The hole condition is hard-enforced by the level-set multiplier, so only the
flux condition remains soft.

Note what that leaves you with: **every** boundary of this problem carries
either a flux condition or a hard-enforced value. Slide 7 is about what breaks
when a problem is pure Neumann; keep it in mind when you read your peak
temperature, and be ready to say why this problem escapes the trap.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08.1-stationary-heat/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · The model and the three point sets

Both `xy_f` and `xy_o` need `requires_grad=True`. The interior points are
obvious — the residual is second derivatives there. The outer edge points are
the one people forget: the zero-flux condition is a **first** derivative of the
network, so those points get differentiated too.

In [ ]:
Q_OVER_K = 10.0
N_COLL = 1500

set_seed(88)
model = MLP(n_in=2, n_hidden=40, n_layers=3)
describe(model, N_COLL)

xy_f = to_tensor(pb.sample_plate_with_hole(N_COLL), requires_grad=True)
xy_o = to_tensor(pb.sample_outer_edges(30), requires_grad=True)
print("interior", tuple(xy_f.shape), " outer", tuple(xy_o.shape))

## 2 · The trial solution

### TODO 1 — trial solution using the hole multiplier

In [ ]:
# TODO 1 --- the trial solution with the hole built in -----------------------------------------------------
# One `...` to replace:  pb.hole_multiplier(xy) * model(xy)
#   hole_multiplier is the level set: zero on the hole, positive in the material; works on tensors
def trial(model, xy):
    return ...                                    # <- pb.hole_multiplier(xy) * model(xy)
# ------------------------------------------------------------------------------

## 3 · Residual, and the zero-flux loss on the outer edges

### TODO 2 — residual, and the zero-flux loss on the outer edges

The outer edges are axis-aligned, so the normal derivative is just the
x- or y-derivative depending on the edge. A simpler and adequate approach:
penalise the full gradient magnitude on those points.

In [ ]:
# TODO 2 --- residual, flux term, loss --------------------------------------------------------------------
# Three `...` to replace:
#   line 1  ->  d2(T, xy, 0) + d2(T, xy, 1) + Q_OVER_K                       T_xx + T_yy + Q/k
#   line 2  ->  (g ** 2).sum(dim=1, keepdim=True).mean()                     insulated outer edges: gradient zero
#   line 3  ->  mse(residual(model, xy_f)) + W_FLUX * flux_loss(model, xy_o)
W_FLUX = 1.0

def residual(model, xy):
    T = trial(model, xy)
    return ...                                    # <- d2(T, xy, 0) + d2(T, xy, 1) + Q_OVER_K

def flux_loss(model, xy):
    T = trial(model, xy)
    g = grad(T, xy)
    # on an axis-aligned edge only the outward component must vanish;
    # penalising both is stricter but acceptable here
    return ...                                    # <- (g ** 2).sum(dim=1, keepdim=True).mean()

def loss_fn():                                    # no arguments: train_two_stage calls loss_fn() and nothing else
    return ...                                    # <- mse(residual(model, xy_f)) + W_FLUX * flux_loss(model, xy_o)
# ------------------------------------------------------------------------------

## 4 · Train, and read the answer like an engineer

A contour plot is not a result. The result is the peak temperature and where it
sits — slide 22.

In [ ]:
history = train_two_stage(model, loss_fn,
                          adam_steps=3000, lbfgs_steps=200, lr=1e-3)

plot_curves(history, title="plate with a cooling hole, w_flux = 1")
plt.show()

In [ ]:
X, Y, T = pb.eval_grid_masked(model, trial=trial)
peak = np.nanmax(T); iy, ix = np.unravel_index(np.nanargmax(T), T.shape)
print(f"peak T = {peak:.4f} at (x, y) = ({X[iy, ix]:.3f}, {Y[iy, ix]:.3f})")

plt.figure(figsize=(5.5, 4.6))
plt.contourf(X, Y, T, 50, cmap="magma"); plt.colorbar()
plt.plot(X[iy, ix], Y[iy, ix], "wo", ms=8)
plt.gca().set_aspect("equal"); plt.title("steady temperature"); plt.show()

## 5 · Save

In [ ]:
os.makedirs("Ex08.1_outputs", exist_ok=True)
path = os.path.join("Ex08.1_outputs", "nb02_hole.npz")
np.savez(path,
         peak=float(peak), loc=np.array([X[iy, ix], Y[iy, ix]]),
         q_over_k=Q_OVER_K, w_flux=W_FLUX, T=T,
         adam=history["adam"], lbfgs=history["lbfgs"])
torch.save(model.state_dict(), os.path.join("Ex08.1_outputs", "nb02_hole.pt"))
print("wrote", path)

**Question.** Where does the peak sit relative to the hole, and why
there?

Next: **notebook 03**, which asks whether the flux condition was learned at all
— and the contour plot above cannot tell you.